# Periodogram Covariance — Gaussian Formula Validation (Space-Time)

## 목적

$$\mathrm{Cov}(I_{\mathbf{j}}, I_{\mathbf{k}}) = |S_{\mathbf{jk}}|^2 + |P_{\mathbf{jk}}|^2$$

이 식이 수학적으로 맞는지 시뮬레이션으로 직접 검증한다.

**방법**:
1. 알려진 space-time covariance (7 params) 로 circulant embedding으로 실제 field 생성
2. **31 독립 day** × 8 슬롯 = **248 snapshots** — empirical 노트북과 동일 구조
3. 각 time slice에 **2-1-1-0 differencing** 적용 → 113×158 grid
4. **Hann taper** 적용, empirical 노트북과 동일 normalization
5. empirical 노트북과 **동일 주파수** (anchor (1,1), row/col sweep, Nyquist)
6. True $f_Z$ (filtered spectral density) 로 Gaussian baseline 계산 후 비교

**비교 가능 조건**: N_SNAP=248, 주파수, taper, differencing 모두 empirical과 동일

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fftn, ifftn
from matplotlib.patches import Patch

# ── Grid & data params (empirical 노트북과 동일) ───────────────────────────────
M_grid, N_grid = 114, 159       # raw grid
M_d,    N_d    = 113, 158       # after 2-1-1-0 differencing
DLAT,   DLON   = 0.044, 0.063
T_STEPS        = 8
N_DAYS         = 31
N_SNAP         = N_DAYS * T_STEPS   # 248
RNG_SEED       = 42

# ── True params (sim_dw_filter_comparison 동일) ───────────────────────────────
true_dict = {
    'sigmasq':    13.059,
    'range_lat':  0.154,
    'range_lon':  0.195,
    'range_time': 1.0,
    'advec_lat':  0.0218,
    'advec_lon': -0.1689,
    'nugget':     0.247,
}
phi2 = 1.0 / true_dict['range_lon']
phi1 = true_dict['sigmasq'] * phi2
phi3 = (true_dict['range_lon'] / true_dict['range_lat'])**2
phi4 = (true_dict['range_lon'] / true_dict['range_time'])**2
true_log = [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4),
            true_dict['advec_lat'], true_dict['advec_lon'], np.log(true_dict['nugget'])]

# ── Hann taper on filtered grid (empirical 노트북 동일) ───────────────────────
h_d   = np.outer(np.hanning(M_d), np.hanning(N_d))
H_q_full = float(np.sum(h_d**2))   # full-grid H_q (no missing obs)
MN_d  = M_d * N_d
MN    = M_grid * N_grid

# ── Frequency bands (empirical sweep_compute 동일) ────────────────────────────
ANCHOR    = (1, 1)
ROW_SWEEP = [(1, j2) for j2 in range(1, 11)]    # (1,1)→(1,10)
COL_SWEEP = [(j1, 1) for j1 in range(1, 11)]    # (1,1)→(10,1)
HIGH_SWEEP= [(55, 78), (55, 79), (56, 78), (56, 79)]   # near Nyquist

ALL_FREQS  = list(dict.fromkeys(ROW_SWEEP + COL_SWEEP[1:] + HIGH_SWEEP))
K          = len(ALL_FREQS)

# band labels for plotting
BAND_LABELS = []
for f in ALL_FREQS:
    if f in ROW_SWEEP and f in COL_SWEEP: BAND_LABELS.append('Anchor')
    elif f in ROW_SWEEP:  BAND_LABELS.append('RowSweep')
    elif f in COL_SWEEP:  BAND_LABELS.append('ColSweep')
    else:                 BAND_LABELS.append('HighFreq')

band_colors_ = {'Anchor': '#333333', 'RowSweep': '#2166ac',
                'ColSweep': '#d6604d', 'HighFreq': '#1a9641'}

print(f'Grid (raw): {M_grid}×{N_grid}  →  filtered: {M_d}×{N_d}')
print(f'N_DAYS={N_DAYS}  T_STEPS={T_STEPS}  N_SNAP={N_SNAP}')
print(f'K={K} frequencies  |  H_q_full={H_q_full:.2f}')
print(f'Freqs: {ALL_FREQS[:5]} ... {ALL_FREQS[-4:]}')


In [2]:
# ── True spectral density: spatial marginal + 2-1-1-0 filter ────────────────
#
#   C_space(h1, h2) = C(h1, h2, Δt=0)  — spatial marginal at lag-0
#   f_X(k1,k2)      = FFT(C_space) on (M,N) grid
#   H_filt(k1,k2)   = -2 + exp(-2πi k1/M) + exp(-2πi k2/N)  — filter TF
#   f_Z_true(k1,k2) = |H_filt|² × f_X   — spectral density of Z = filtered X
#
#   Note: filtered grid is (M_d,N_d)=(M-1,N-1); spectral density evaluated at
#         same integer freq indices on the full (M,N) grid (approx valid for
#         inner frequencies, fine for k << M/2)
# ─────────────────────────────────────────────────────────────────────────────

i_idx = np.arange(M_grid); i_idx[M_grid//2:] -= M_grid
j_idx = np.arange(N_grid); j_idx[N_grid//2:] -= N_grid
H_lat, H_lon = np.meshgrid(i_idx * DLAT, j_idx * DLON, indexing='ij')

dist      = np.sqrt(phi3 * H_lat**2 + H_lon**2 + 1e-12)
C_space   = (phi1 / phi2) * np.exp(-phi2 * dist)
C_space[0, 0] += true_dict['nugget']

f_X = np.real(fftn(C_space))
f_X = np.clip(f_X, 0, None)

# 2-1-1-0 filter transfer function on (M,N) grid
k1_idx = np.arange(M_grid)[:, None]
k2_idx = np.arange(N_grid)[None, :]
H_filt = -2.0 + np.exp(-2j * np.pi * k1_idx / M_grid) + np.exp(-2j * np.pi * k2_idx / N_grid)
f_Z_true = (np.abs(H_filt)**2) * f_X

# H_dft for taper (on filtered grid M_d×N_d for Gaussian baseline)
H_dft_d = fftn(h_d)   # shape (M_d, N_d)

print(f"C_space(0,0) = {C_space[0,0]:.4f}  (sigmasq+nugget = {true_dict['sigmasq']+true_dict['nugget']:.4f})")
print(f"f_X   : min={f_X.min():.4f}  max={f_X.max():.4f}")
print(f"f_Z   : min={f_Z_true.min():.4f}  max={f_Z_true.max():.4f}")
print(f"\n{'#':>3}  {'Band':>9}  {'freq':>10}  {'f_X':>10}  {'|H_filt|²':>11}  {'f_Z_true':>10}")
print('-'*58)
for q, (j, band) in enumerate(zip(ALL_FREQS, BAND_LABELS)):
    fx = f_X[j[0], j[1]]
    hf = abs(H_filt[j[0], j[1]])**2
    fz = f_Z_true[j[0], j[1]]
    print(f"  {q:2d}  {band:9s}  ({j[0]:2d},{j[1]:2d})  {fx:10.3f}  {hf:11.6f}  {fz:10.4f}")


In [3]:
# ── Circulant embedding field generation (sim_dw_filter_comparison 방식) ────
#
#   generate_day_field: (M, N, T) space-time field via 3D circulant embedding
#     Px,Py,Pt = 2M,2N,2T  (embedding dimension)
#     C_3d[lx,ly,lt] = (phi1/phi2)*exp(-phi2*||(u_lat,u_lon)||) + nugget*δ
#     S = FFT_3d(C_3d),  S.real clamped ≥ 0
#     noise = FFT_3d(randn(Px,Py,Pt))
#     field = IFFT_3d(sqrt(S) * noise).real[:M,:N,:T]
# ─────────────────────────────────────────────────────────────────────────────

def generate_day_field(M, N, T, tlog, dlat, dlon, rng_):
    ep1, ep2, ep3, ep4 = [np.exp(tlog[i]) for i in range(4)]
    alat, alon, nugget = tlog[4], tlog[5], np.exp(tlog[6])
    Px, Py, Pt = 2*M, 2*N, 2*T

    lx = np.arange(Px) * dlat;  lx[Px//2:] -= Px * dlat
    ly = np.arange(Py) * dlon;  ly[Py//2:] -= Py * dlon
    lt = np.arange(Pt, dtype=float); lt[Pt//2:] -= Pt

    Lx, Ly, Lt = np.meshgrid(lx, ly, lt, indexing='ij')
    u_lat = Lx - alat * Lt
    u_lon = Ly - alon * Lt
    dist  = np.sqrt(u_lat**2 * ep3 + u_lon**2 + Lt**2 * ep4 + 1e-12)
    C     = (ep1 / ep2) * np.exp(-ep2 * dist)
    C[0, 0, 0] += nugget

    S = np.real(np.fft.fftn(C))
    S = np.clip(S, 0, None)

    noise = np.fft.fftn(rng_.standard_normal((Px, Py, Pt)))
    field = np.fft.ifftn(np.sqrt(S) * noise).real[:M, :N, :T]
    return field * np.sqrt(Px * Py * Pt)   # scale: Var → C(0,0,0)

# ── Simulation: N_DAYS independent days ──────────────────────────────────────
rng = np.random.default_rng(RNG_SEED)

J_tap_sim = np.zeros((N_SNAP, K), dtype=complex)
J_raw_sim = np.zeros((N_SNAP, K), dtype=complex)
f_plug_d  = np.zeros((M_d, N_d), dtype=float)
norm_full  = 1.0 / (2.0 * np.pi * np.sqrt(H_q_full))
scale_raw  = np.sqrt(MN_d)

print(f"Generating {N_DAYS} days × {T_STEPS} slices = {N_SNAP} snapshots ...")
snap_idx = 0
for d in range(N_DAYS):
    field = generate_day_field(M_grid, N_grid, T_STEPS, true_log, DLAT, DLON, rng)
    # field shape: (M_grid, N_grid, T_STEPS)

    for t in range(T_STEPS):
        X = field[:, :, t]
        X = X - X.mean()

        # 2-1-1-0 differencing → (M_d, N_d)
        Z = -2.0*X[:-1, :-1] + X[1:, :-1] + X[:-1, 1:]

        # Hann taper (no missing obs in simulation → g_eff = h_d)
        F_tap = fftn(h_d * Z)
        F_raw = fftn(Z)

        f_plug_d += np.abs(F_tap * norm_full)**2

        for q, (j1, j2) in enumerate(ALL_FREQS):
            J_tap_sim[snap_idx, q] = F_tap[j1, j2] * norm_full
            J_raw_sim[snap_idx, q] = F_raw[j1, j2] / scale_raw
        snap_idx += 1

    if (d+1) % 5 == 0:
        print(f"  Day {d+1}/{N_DAYS} done")

f_plug_d /= N_SNAP

I_tap_sim  = np.abs(J_tap_sim)**2
I_raw_sim  = np.abs(J_raw_sim)**2
f_hat_tap  = I_tap_sim.mean(axis=0)
f_hat_raw  = I_raw_sim.mean(axis=0)

def emp_cov(I):
    Ic = I - I.mean(axis=0)
    return (Ic.T @ Ic) / I.shape[0]

Cov_emp_tap = emp_cov(I_tap_sim)
Cov_emp_raw = emp_cov(I_raw_sim)

print(f"\nDone.")
print(f"\n{'q':>3}  {'Band':>9}  {'freq':>10}  {'f_Z_true':>10}  {'f_hat':>10}  {'ratio':>7}")
print('-'*58)
for q, (j, band) in enumerate(zip(ALL_FREQS, BAND_LABELS)):
    fz = f_Z_true[j[0], j[1]]
    fh = f_hat_tap[q]
    print(f"  {q:2d}  {band:9s}  ({j[0]:2d},{j[1]:2d})  {fz:10.4f}  {fh:10.4f}  {fh/(fz+1e-15):7.3f}")


In [4]:
# ── Gaussian baselines ───────────────────────────────────────────────────────
#
#   Cov(I_j, I_k) = |S_jk|² + |P_jk|²
#   S_jk = (1/(H_q * MN_d)) Σ_m f(ω_m) · H_dft[j-m] · H_dft*[k-m]
#   P_jk = (1/(H_q * MN_d)) Σ_m f(ω_m) · H_dft[j-m] · H_dft[k+m]
#
#   A. f = f_Z_true  (true filtered spectral density) — formula check
#   B. f = f_plug_d  (plug-in from simulated periodograms)
#   C. Smooth (Priestley approx)
# ─────────────────────────────────────────────────────────────────────────────
m1 = np.arange(M_d)[:, None]
m2 = np.arange(N_d)[None, :]

def exact_cov_gauss(f_spectrum):
    C = np.zeros((K, K))
    for i, (j1, j2) in enumerate(ALL_FREQS):
        H_j = H_dft_d[(j1-m1)%M_d, (j2-m2)%N_d]
        for l, (k1, k2) in enumerate(ALL_FREQS):
            H_kc = np.conj(H_dft_d[(k1-m1)%M_d, (k2-m2)%N_d])
            H_kp =         H_dft_d[(k1+m1)%M_d, (k2+m2)%N_d]
            S_jk = np.sum(f_spectrum[(slice(None),)*2] * H_j * H_kc) / (H_q_full * MN_d)
            P_jk = np.sum(f_spectrum[(slice(None),)*2] * H_j * H_kp) / (H_q_full * MN_d)
            C[i, l] = abs(S_jk)**2 + abs(P_jk)**2
    return C

# f_Z_true is (M_grid, N_grid) but H_dft_d is (M_d, N_d) — crop to (M_d, N_d)
f_Z_crop  = f_Z_true[:M_d, :N_d]
f_plug_use = f_plug_d   # (M_d, N_d)

print("Computing Gaussian baselines ...")
Cov_gauss_true   = exact_cov_gauss(f_Z_crop)
print("  A. Exact(f_Z_true) done")
Cov_gauss_plugin = exact_cov_gauss(f_plug_use)
print("  B. Exact(f_plug)   done")

W_g2_d = fftn(h_d**2)
Cov_gauss_smooth = np.zeros((K, K))
for i, (j1,j2) in enumerate(ALL_FREQS):
    for l, (k1,k2) in enumerate(ALL_FREQS):
        dm = ((j1-k1)%M_d, (j2-k2)%N_d)
        dp = ((j1+k1)%M_d, (j2+k2)%N_d)
        Cov_gauss_smooth[i,l] = (f_hat_tap[i]*f_hat_tap[l]
            * (abs(W_g2_d[dm])**2 + abs(W_g2_d[dp])**2) / H_q_full**2)
print("  C. Smooth(Priestley) done")

print(f"\n{'q':>3}  {'Band':>9}  {'freq':>10}  "
      f"{'Var_emp':>10}  {'True':>10}  {'Plugin':>10}  {'Smooth':>10}")
print('-'*68)
for q, (j,band) in enumerate(zip(ALL_FREQS, BAND_LABELS)):
    print(f"  {q:2d}  {band:9s}  ({j[0]:2d},{j[1]:2d})  "
          f"{Cov_emp_tap[q,q]:10.4e}  {Cov_gauss_true[q,q]:10.4e}  "
          f"{Cov_gauss_plugin[q,q]:10.4e}  {Cov_gauss_smooth[q,q]:10.4e}")


In [5]:
# ── Correlation matrix: Emp vs Gaussian baselines ────────────────────────────
def to_corr(C):
    std = np.sqrt(np.clip(np.diag(C), 1e-30, None))
    return C / np.outer(std, std)

def draw_mat(ax, mat, cmap, vmin, vmax, title):
    im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    ax.set_title(title, fontsize=8, fontweight='bold')
    tlbls = [f'({j[0]},{j[1]})' for j in ALL_FREQS]
    tcolors = [band_colors_[b] for b in BAND_LABELS]
    ax.set_xticks(range(K)); ax.set_yticks(range(K))
    ax.set_xticklabels(tlbls, rotation=45, ha='right', fontsize=5)
    ax.set_yticklabels(tlbls, fontsize=5)
    for q, tc in enumerate(tcolors):
        ax.get_xticklabels()[q].set_color(tc)
        ax.get_yticklabels()[q].set_color(tc)
    # separators: RowSweep=10, ColSweep=9, HighFreq=4
    for sep in [9.5, 18.5]:
        ax.axhline(sep, color='gray', lw=0.6, ls='--')
        ax.axvline(sep, color='gray', lw=0.6, ls='--')
    for i in range(K):
        for l in range(K):
            ax.text(l, i, f'{mat[i,l]:.2f}', ha='center', va='center',
                    fontsize=4, color='white' if abs(mat[i,l])>0.5 else 'black')

Cemp  = to_corr(Cov_emp_tap)
Ctrue = to_corr(Cov_gauss_true)
Cplug = to_corr(Cov_gauss_plugin)
Csmth = to_corr(Cov_gauss_smooth)
Cemp_raw = to_corr(Cov_emp_raw)

rows = [
    (Cemp,     Ctrue, 'Exact(f_Z_true) — FORMULA CHECK', 'Gauss_true'),
    (Cemp,     Cplug, 'Exact(plug-in)  — PLUG-IN BIAS',  'Gauss_plugin'),
    (Cemp,     Csmth, 'Smooth (Priestley)',               'Gauss_smooth'),
    (Cemp_raw, np.eye(K), 'NO TAPER — exact diagonal',   'Gauss_raw'),
]

fig, axes = plt.subplots(4, 3, figsize=(21, 28))
fig.suptitle(
    f"SIM (space-time circulant, 2-1-1-0 filter)  |  N={N_SNAP} snapshots ({N_DAYS} days×{T_STEPS})\n"
    f"Freq: anchor (1,1), row/col sweep → (10,·)/(·,10), Nyquist corner",
    fontsize=11, fontweight='bold', y=1.005)

for ri, (Ce, Cg, row_tag, gname) in enumerate(rows):
    res  = Ce - Cg
    rlim = max(0.05, np.abs(res).max())
    draw_mat(axes[ri,0], Ce,       'RdBu_r', -1, 1, f'Empirical Corr [{row_tag}]')
    draw_mat(axes[ri,1], Cg,       'RdBu_r', -1, 1, f'Gaussian ({gname})')
    draw_mat(axes[ri,2], res,      'PiYG', -rlim, rlim,
             f'Residual [{gname}]  max={rlim:.3f}')

handles = [Patch(color=c, label=b) for b,c in band_colors_.items()]
axes[0,0].legend(handles=handles, loc='lower right', fontsize=7, title='Band')
plt.tight_layout(h_pad=4, w_pad=2)
plt.savefig('/tmp/cov_sim_spacetime.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved: /tmp/cov_sim_spacetime.png')


In [6]:
# ── Row / Col sweep: Corr(I(anchor), I(ω_k)) vs separation ─────────────────
anchor_idx = ALL_FREQS.index(ANCHOR)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(
    f"SIM: Corr(I(anchor), I(ω_k))  |  anchor={ANCHOR}  |  N={N_SNAP}\n"
    f"Left=Hann taper  Right=No taper  |  공간주파수 독립성 검증",
    fontsize=11, fontweight='bold')

for row_ax, (sweep, label, get_sep) in enumerate([
    (ROW_SWEEP, 'Row sweep  (j1=1 fixed, j2=1→10)', lambda f: f[1]),
    (COL_SWEEP, 'Col sweep  (j2=1 fixed, j1=1→10)', lambda f: f[0]),
]):
    idxs   = [ALL_FREQS.index(f) for f in sweep]
    seps   = [get_sep(f) for f in sweep]
    corr_t = [Cemp[anchor_idx, i] for i in idxs]
    corr_r = [Cemp_raw[anchor_idx, i] for i in idxs]

    for col_ax, (corr_vals, title_tag, color) in enumerate([
        (corr_t, 'Hann taper', '#2166ac'),
        (corr_r, 'No taper',   '#d73027'),
    ]):
        ax = axes[row_ax, col_ax]
        ax.plot(seps, corr_vals, 'o-', color=color, lw=2, ms=7)
        ax.axhline(0, color='k', lw=0.8)
        ax.set_xlabel('Frequency index', fontsize=10)
        ax.set_ylabel('Corr_emp (SIM)', fontsize=10)
        ax.set_title(f'{label} [{title_tag}]', fontsize=10, fontweight='bold')
        ax.set_xticks(seps)
        ax.set_xticklabels([str(f) for f in sweep], rotation=40, ha='right', fontsize=8)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        ax.yaxis.grid(True, ls='--', alpha=0.35); ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('/tmp/cov_sim_sweep.png', dpi=130, bbox_inches='tight')
plt.show()

# ── 수치 출력 ──────────────────────────────────────────────────────────────────
for sweep, tag, get_sep in [
    (ROW_SWEEP, 'Row (j1=1 fixed)', lambda f: f[1]),
    (COL_SWEEP, 'Col (j2=1 fixed)', lambda f: f[0]),
]:
    print(f"\n  --- {tag} ---")
    print(f"  {'Freq':>10}  {'sep':>4}  {'Corr_tap':>10}  {'Corr_raw':>10}")
    print(f"  {'-'*44}")
    for f in sweep:
        i   = ALL_FREQS.index(f)
        sep = get_sep(f)
        ct  = Cemp[anchor_idx, i]
        cr  = Cemp_raw[anchor_idx, i]
        print(f"  {str(f):>10}  {sep:>4}  {ct:>10.4f}  {cr:>10.4f}")


In [7]:
# ── Formula validation summary ───────────────────────────────────────────────
var_ratio = np.array([Cov_emp_tap[q,q] / (Cov_gauss_true[q,q]+1e-30) for q in range(K)])
offdiag_emp  = [abs(Cemp[i,l])             for i in range(K) for l in range(K) if i!=l]
offdiag_true = [abs(Ctrue[i,l])            for i in range(K) for l in range(K) if i!=l]
offdiag_diff = [abs(Cemp[i,l]-Ctrue[i,l]) for i in range(K) for l in range(K) if i!=l]

print('='*65)
print('  FORMULA VALIDATION SUMMARY')
print('='*65)
print(f'  N_DAYS={N_DAYS}  T_STEPS={T_STEPS}  N_SNAP={N_SNAP}')
print()
print(f'  [Diagonal]  Var_emp / Cov_gauss(f_Z_true):')
print(f'  mean={var_ratio.mean():.3f}  min={var_ratio.min():.3f}  max={var_ratio.max():.3f}')
print(f'  (should → 1.0 as N_DAYS → ∞)')
print()
print(f'  [Off-diagonal]')
print(f'  max |Corr_emp|             = {max(offdiag_emp):.4f}')
print(f'  max |Corr_gauss(f_Z_true)| = {max(offdiag_true):.4f}')
print(f'  max |Corr_emp - Corr_true| = {max(offdiag_diff):.4f}')
print()
print(f'  [High-freq pairs near Nyquist]')
hi_idxs = [ALL_FREQS.index(f) for f in HIGH_SWEEP]
pairs_hi = [(0,1,'(55,78)↔(55,79) Δj2=1'),
            (0,2,'(55,78)↔(56,78) Δj1=1'),
            (0,3,'(55,78)↔(56,79) diag')]
print(f"  {'Pair':>35}  {'|Corr_tap|':>11}  {'|Corr_raw|':>11}")
print(f"  {'-'*62}")
for a,b,tag in pairs_hi:
    ai,bi = hi_idxs[a], hi_idxs[b]
    print(f"  {tag:>35}  {abs(Cemp[ai,bi]):>11.4f}  {abs(Cemp_raw[ai,bi]):>11.4f}")
print('='*65)
